In [90]:
# Search_Names_POJ.py
# 6/1/2025

from bs4 import BeautifulSoup 
import datetime
from datetime import datetime as dt
from datetime import date 
from datetime import timedelta
import time
import re
from pandas import read_csv
import pandas as pd
# from selenium import webdriver
from seleniumwire import webdriver
from selenium.webdriver.common.by import By
import json
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
import sys
import random
from zenrows import ZenRowsClient
import requests
import math
import numpy as np
import urllib3
import inspect
from geopy.distance import distance
# from IPython.display import display


# def maximize_screen():
#     from IPython.display import display, HTML
#     display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

def left(s, amount):
    return s[:amount]
def right(s, amount):
    return s[-amount:]
def mid(s, offset, amount):
    return s[offset:offset+amount]

def maximize_screen():
    from IPython.display import display, HTML
    display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

def timestamp():
    today=str(datetime.date.today()).split('-')
    year=today[0]
    month=today[1]
    day=today[2]
    a=str(datetime.datetime.now())
    time_split=re.split(':| ',a)
    hour=time_split[1]
    timestamp=year+month+day+hour
    return timestamp

def convert_date_z(date): 
    date_t= pd.to_datetime(date, utc=True)
    date_out=date_t.dt.strftime("%m/%d/%Y")
    return date_out

def convert_datetime_t(event_date_t):
    no_time='2000-01-01T00:00:00'
    no_time_dt=dt.strptime(no_time, "%Y-%m-%dT%H:%M:%S") 
    event_date_t = event_date_t.fillna(no_time)
    event_date_dt=pd.to_datetime(event_date_t, format="%Y-%m-%dT%H:%M:%S")
    event_date_index=event_date_dt.dt.strftime("%Y%m%d")
    event_date =event_date_dt.dt.strftime("%m/%d/%Y")
    event_date_rnd_dt = event_date_dt + timedelta(minutes=30)
    event_time_index=event_date_rnd_dt.dt.strftime('%H')  
    event_time = event_date_dt.dt.strftime('%H:%M')
    return     event_date,event_date_index,event_time,event_time_index 



def random_sleep(call_count): 
    global sleep_time_pause,sleep_range
    if call_count%100==0:
        sleep_time =sleep_time_pause
        now_date = datetime.datetime.now() + timedelta(seconds=sleep_time)
        sleep_end_date=now_date.strftime("%m/%d/%Y %H:%M:%S")    
        print(f"\t\t\tSleeping for {sleep_time:.0f} seconds • Sleeping till: {sleep_end_date}")
    else:
        sleep_time = random.uniform(sleep_range[0],sleep_range[1])
    return sleep_time 

def get_total_cemeteries(url):
    global my_header
    response = requests.request("GET", url, headers=my_header) 
    status_code=response.status_code
    url_source = response.text 
    
    beautifulSoupText = BeautifulSoup(url_source, 'html.parser') 
    specific_tag_text = beautifulSoupText.get_text()
    capture_json=json.loads(specific_tag_text) 
    total_cemeteries=capture_json['data']['cemeteries']['total']
    return total_cemeteries

def get_grave_count(url):
    response = requests.request("GET", url, headers=my_header) 
    status_code=response.status_code
    url_source = response.text 
    beautifulSoupText = BeautifulSoup(url_source, 'html.parser') 
    specific_tag_text = beautifulSoupText.get_text()
    capture_json=json.loads(specific_tag_text) 
    grave_count=capture_json['total']
    return capture_json,grave_count
    
def extract_grave_data(capture_json):
    global field_list_set,field_list
    collection_df=pd.json_normalize(capture_json['collection'])
    collection_columns_set=set(list(collection_df))
    missing_columns=field_list_set-collection_columns_set
    collection_df[list(missing_columns)]=np.nan
    grave_frame_df=collection_df[field_list]
    return grave_frame_df

def clean_dataframe(graves_df):
    global field_list
    for column in field_list:
        graves_df[column]= graves_df[column].astype(str).replace(r"\,", ' * ', regex=True).str.strip()
        graves_df[column]= graves_df[column].astype(str).replace(r"\'", '', regex=True).str.strip()
    delete_list=[r'\"',r"\'",r'\?',r'\[',r'\]']
    for item in delete_list:
        for column in field_list:
            graves_df[column]= graves_df[column].astype(str).replace(item, ' ', regex=True).str.strip()
    return graves_df


def get_cemetery_cluster(name_location):
    global jewish_cemetery_df,non_jewish_cemetery_df,county_coordinates_df
    county_distance_df=pd.DataFrame([])
    # name_location=[name_lat,name_lon]
    for _, row in county_coordinates_df.iterrows():
        county_distance=distance(name_location, (row['county_lat'], row['county_lon'])).miles
        county_distance_line_df=pd.DataFrame([row['county_id'],row['county_name'],row['state_name'],county_distance]).T
        county_distance_line_df.columns=['county_id','county_name','state_name','distance']
        county_distance_df=pd.concat([county_distance_df,county_distance_line_df])
    cluster_county_id_df=county_distance_df[['county_id','county_name','state_name']][county_distance_df['distance']<=jewish_radius]
    matched_cemeteries_df_header=['type','cemetery_id','cemetery_name','grave_count','city','state','stateAbbreviation',\
                              'county','state_id','county_id','cemetery_lat','cemetery_lon','County_Cemetery_Counts']
    jewish_cemetery_df=pd.merge(
        jewish_cemetery_df,    
        cluster_county_id_df,
        right_on='county_id',             
        left_on='county_id',
        how='inner'  # or 'left', 'right', 'outer' depending on what you want
    )
    jewish_cemetery_df['type']='Jewish'
    
    non_jewish_cemetery_df=pd.merge(
        non_jewish_cemetery_df,    
        cluster_county_id_df,
        right_on='county_id',             
        left_on='county_id',
        how='inner'  # or 'left', 'right', 'outer' depending on what you want
    )
    non_jewish_cemetery_df['type']='Non_Jewish'
    matched_cemeteries_df=pd.DataFrame([])
    matched_cemeteries_df = non_jewish_cemetery_df._append(jewish_cemetery_df, ignore_index = True) 
    matched_cemeteries_df=matched_cemeteries_df[matched_cemeteries_df_header]
    return matched_cemeteries_df,cluster_county_id_df

def find_county_coordinates(name_county_id):
    global county_coordinates_df,first_name,last_name
    name_lat=0
    name_lon=0
    try:
        name_coordiantes=county_coordinates_df[['county_lat','county_lon']][county_coordinates_df['county_id']==name_county_id]
        name_lat=name_coordiantes.iloc[0,0]
        name_lon=name_coordiantes.iloc[0,1]
    except:
        print(f" XXXXX {last_name},{first_name}  ***** No County Found *****   XXXXX")
    return name_lat,name_lon

my_header = {
  'priority': 'u=1, i',
  'sec-ch-ua': '"Google Chrome";v="135", "Not-A.Brand";v="8", "Chromium";v="135"',
  'sec-ch-ua-platform': '"Windows"',
  'sec-fetch-mode': 'cors',
  'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36',
  'Cookie': '__cf_bm=O55WB_mmMt72i8KoQuzWliizNTb6J7zg2RVHXqbn.vc-1745620049-1.0.1.1-IDSlPNW4kRjBMAr5n8M2obOoGp9GAod_BHbB997Ze2F0YUm2DvAFh5TaijsQhPWR0iyIyFoaSy.p1YDke07ZttHHC83SJpQCEBFmaeImKIc; _cfuvid=zhvP.IczaAr9sd4YVjWjTqBayGxsei_Ntl0ANgSCCBg-1745620049033-0.0.1.1-604800000; ipLoc=us; preferredLanguage=en'
}

#*************** Append  *******************************
append=0
# append_flag=input('To Append Press 1')
# if append_flag=='1':append=1
#*************** Append  *******************************

maximize_screen()
timestamp=timestamp()
collection_date0=datetime.datetime.now().strftime("%m/%d/%Y")
collection_date_dt=pd.to_datetime(collection_date0)
collection_date=collection_date0

start_time=datetime.datetime.now()
event_time=start_time
blank_line='                                                                                                                                                                               '

process_name='Extract POJ from ABMC Names '
print('**Starting *** ',process_name,' *** , Time:',start_time,' **\n')
DirMain="C:\\Users\\jrozm\\OneDrive\\Benjamin\\"

fileFirstLastIn="First_Last_Names_List"
filePOJOut="Names_POJ"

fileIn=DirMain+"Analysis\\"+fileFirstLastIn+".csv"
fileOutPOJ=DirMain+"Analysis\\SearchResults\\"+filePOJOut+"_"+timestamp+".csv"
fileOutPOJ0=DirMain+"Analysis\\SearchResults\\"+filePOJOut+".csv"
fileOutPOJInt=DirMain+"Analysis\\SearchResults\\"+filePOJOut+"_Interim.csv"

fileGravesOut="Names_Graves"

fileOutGraves=DirMain+"Analysis\\SearchResults\\"+fileGravesOut+"_"+timestamp+".csv"
fileOutGraves0=DirMain+"Analysis\\SearchResults\\"+fileGravesOut+".csv"
fileOutGravesInt=DirMain+"Analysis\\SearchResults\\"+fileGravesOut+"_Interim.csv"

fileJewishIn="Jewish_Cluster_Cemetery_List"
fileNonJewishIn="NonJewish_Cluster_Cemetery_List"
fileCountyIn="Jewish_Counties_Coordinates"
fileCountyIn="Benjamin_Counties"

fileInJ=DirMain+"Data\\"+fileJewishIn+".csv"
fileInN=DirMain+"Data\\"+fileNonJewishIn+".csv"
fileInCounty=DirMain+"Data\\"+fileCountyIn+".csv"

fileCountyIn="Benjamin_Counties"



jewish_cemetery_df=pd.read_csv(fileInJ)
non_jewish_cemetery_df=pd.read_csv(fileInN)
county_coordinates_df=pd.read_csv(fileInCounty)

cemetery_field_list=['cemetery_id','grave_count','cemetery_name','city','county	state','stateAbbreviation','county_id','state_id','cemetery_lat','cemetery_lon']
 
grave_df_header_final=['type','lastName','firstName','fullName','maidenName','birthYear','deathYear','birthDate','deathDate','cemeteryName',\
                       'cemeteryId','cityName','stateName','distance','Parents','Children','Siblings','Spouses']
field_list=grave_df_header_final

summary_df_header=['Name','City','State','Total_Matched_Graves','Jewish_Graves','Non_Jewish_Graves','Jewish_Graves_Percent','Jewish_Radius','Non_Jewish_Radius']
matched_cemeteries_df_header=['type','cemetery_id','cemetery_name','distance','city','state','state_id','county_id','lat','lon','grave_count']

first_last_names_df=pd.read_csv(fileIn)

first_last_names_df["last_name"]=first_last_names_df["last_name"].str.lower()
first_last_names_df["first_name"]=first_last_names_df["first_name"].str.lower()
first_last_name_list=first_last_names_df[["last_name","first_name"]]
name_lat_list=first_last_names_df["lat"]
name_lon_list=first_last_names_df["lon"]
name_city_list=first_last_names_df["City"]
name_state_list=first_last_names_df["State"]
name_county_id_list=first_last_names_df["county_id"]
name_county_name_list=first_last_names_df["county_name"]



field_list=['lastName','firstName','fullName','birthYear','deathYear','birthMonth','deathMonth','birthDate','deathDate',\
    'cemeteryName','cemeteryId','cityName','stateName','maidenName','Parents','Children', 'Siblings','Spouses']
field_list_set=set(field_list)
if append==1:print('**** Appending ****')    
if append!=1:graves_df=pd.DataFrame([])  

death_year=1940
birth_year=1920
search_type='death'
search_type='birth'


if search_type=='death':
    url0 = "https://www.findagrave.com/memorial/search?ajax=true&skip="
    url1 = "&limit=100&firstname=&middlename=&lastname="
    url2 ="&birthyear=&birthyearfilter=&deathyear="+str(death_year)+"&deathyearfilter=before&locationId="
    url3="&includeMaidenName=false"
    
    url0_f = "https://www.findagrave.com/memorial/search?ajax=true&skip="
    url1_f = "&limit=100&firstname="
    url2_f ="&middlename=&lastname=&exactName=true&birthyear=&birthyearfilter=&deathyear="+str(death_year)+"&deathyearfilter=before&locationId="
    url3_f="&includeMaidenName=false"
    search_year=death_year
else:
    url0 = "https://www.findagrave.com/memorial/search?ajax=true&skip="
    url1 = "&limit=100&firstname=&middlename=&lastname="
    url2 ="&birthyear="+str(birth_year)+"&birthyearfilter=before&deathyear=&deathyearfilter=&locationId="
    url3="&includeMaidenName=false"
    
    url0_f = "https://www.findagrave.com/memorial/search?ajax=true&skip="
    url1_f = "&limit=100&firstname="
    url2_f ="&middlename=&lastname=&exactName=true&birthyear="+str(birth_year)+"&birthyearfilter=before&deathyear=&deathyearfilter=&locationId="
    url3_f="&includeMaidenName=false"
    search_year=birth_year
    

summary_df=pd.DataFrame([])
sleep_range=[3,8]
sleep_time_pause=1*60

jewish_radius=25
non_jewish_radius=jewish_radius
get_counter=0
name_count=0    


name_counts_df=pd.DataFrame([])
first_last_name_df=pd.DataFrame([])
name_df=pd.DataFrame([])
for index, row in first_last_name_list.iterrows():
    first_name = row['first_name']
    last_name = row['last_name']    
    name_state=name_state_list.iloc[name_count]
    name_city=name_city_list.iloc[name_count]
    name_county_id=name_county_id_list.iloc[name_count]
    name_county_name=name_county_name_list.iloc[name_count]
    name_lat,name_lon=find_county_coordinates(name_county_id)  
    name_count+=1
    name_location=[name_lat,name_lon]
    matched_cemeteries_df,county_list=get_cemetery_cluster(name_location)   
    county_id_list=county_list['county_id']
    total_county_count=len(county_id_list)

    print(f"•••• Searching: {last_name},{first_name} • Search By: {search_type} • Before: {search_year} • Radius: {jewish_radius} miles • Counties in Cluster: {total_county_count}")
    print(county_list,'\n')
    county_search_count=0
    county_count=0    
    for county_id in county_id_list:
        last_jewish_graves_last_name=0
        last_non_jewish_graves_last_name =0
        last_jewish_graves_first_name=0
        last_non_jewish_graves_first_name =0      
        county_name=county_list['county_name'].iloc[county_count]
        state_name=county_list['state_name'].iloc[county_count]
        county_count+=1
        frame_count=0
        grave_count=100000
        grave_frame_count=0
        frame_count=0
        while grave_frame_count<grave_count:
            frame_count+=1
            url=url0+str(grave_frame_count)+url1+last_name+url2+county_id+url3
            capture_json,grave_count=get_grave_count(url)
            last_name_frame_df_raw=extract_grave_data(capture_json)
            grave_frame_count+=len(last_name_frame_df_raw) 
            last_name_frame_df_raw['cemeteryId']=last_name_frame_df_raw['cemeteryId'].fillna(0).astype(int)
            last_name_frame_df_raw['lastName']=last_name_frame_df_raw['lastName'].astype(str).str.lower()
            last_name_frame_df_raw['firstName']=last_name_frame_df_raw['firstName'].astype(str).str.lower()
            last_name_frame_df_raw = pd.merge(
                    last_name_frame_df_raw,    
                    matched_cemeteries_df,
                    right_on='cemetery_id',             
                    left_on='cemeteryId',
                    how='inner'  # or 'left', 'right', 'outer' depending on what you want
                                                )
            print(f" {blank_line}", end='\r')                                  
            print(f"    .... Last Name:   {last_name:<10} • {county_name:<20} • {state_name:<15} • {county_id} • Grave Counts: {grave_frame_count}/{grave_count}",end='\r')
            last_name_frame_df=clean_dataframe(last_name_frame_df_raw)                               
            name_df= pd.concat([name_df,last_name_frame_df])
            get_counter+=1
            sleep_time=random_sleep(get_counter)       
            time.sleep(sleep_time)
        jewish_graves_last_name=((last_name_frame_df['type']=='Jewish') & ((last_name_frame_df['lastName']==last_name))).sum()
        non_jewish_graves_last_name=((last_name_frame_df['type']=='Non_Jewish')  & ((last_name_frame_df['lastName']==last_name))).sum()
        jewish_graves_last_name_frame=jewish_graves_last_name-last_jewish_graves_last_name
        non_jewish_graves_last_name_frame=non_jewish_graves_last_name-last_non_jewish_graves_last_name  
        last_jewish_graves_last_name=jewish_graves_last_name
        last_non_jewish_graves_last_name =non_jewish_graves_last_name   
        print(f" {blank_line}", end='\r') 
        print(f"    .... Last Name:   {last_name:<10} • {county_name:<20} • {state_name:<15} • {county_id}  • Name Counts - Jewish/Non Jewish: {jewish_graves_last_name_frame}/{non_jewish_graves_last_name_frame}", end='\r')
        # if jewish_graves_last_name+non_jewish_graves_last_name==0:continue
#*************** First Name Search *******************************            
        frame_count=0
        grave_count=100000
        grave_frame_count=0
        frame_count=0
        while grave_frame_count<grave_count:
            frame_count+=1
            url=url0_f+str(grave_frame_count)+url1_f+first_name+url2_f+county_id+url3_f
            capture_json,grave_count=get_grave_count(url)
            first_name_frame_df_raw=extract_grave_data(capture_json)
            grave_frame_count+=len(first_name_frame_df_raw) 
            first_name_frame_df_raw['cemeteryId']=first_name_frame_df_raw['cemeteryId'].fillna(0).astype(int)
            first_name_frame_df_raw['lastName']=first_name_frame_df_raw['lastName'].astype(str).str.lower()
            first_name_frame_df_raw['firstName']=first_name_frame_df_raw['firstName'].astype(str).str.lower()
            first_name_frame_df_raw = pd.merge(
                    first_name_frame_df_raw,    
                    matched_cemeteries_df,
                    right_on='cemetery_id',             
                    left_on='cemeteryId',
                    how='inner'  # or 'left', 'right', 'outer' depending on what you want
                                                )
            print(f" {blank_line}", end='\r')                                
            print(f"    .... First Name:   {first_name:<10} • {county_name:<20} • {state_name:<15} • {county_id} • Grave Counts: {grave_frame_count}/{grave_count}", end='\r')
            first_name_frame_df=clean_dataframe(first_name_frame_df_raw)                                                 
            name_df= pd.concat([name_df,first_name_frame_df])
            get_counter+=1
            sleep_time=random_sleep(get_counter)       
            time.sleep(sleep_time)

        jewish_graves_first_name=((first_name_frame_df['type']=='Jewish') & ((first_name_frame_df['firstName']==first_name))).sum()
        non_jewish_graves_first_name=((first_name_frame_df['type']=='Non_Jewish')  & ((first_name_frame_df['firstName']==first_name))).sum()
        jewish_graves_first_name_frame=jewish_graves_first_name-last_jewish_graves_first_name
        non_jewish_graves_first_name_frame=non_jewish_graves_first_name-last_non_jewish_graves_first_name  
        last_jewish_graves_first_name=jewish_graves_first_name
        last_non_jewish_graves_first_name =non_jewish_graves_first_name  
        if  (jewish_graves_first_name*jewish_graves_last_name+non_jewish_graves_first_name*non_jewish_graves_last_name)==0:
            poj=0
        else:
            poj=int(100*(jewish_graves_first_name*jewish_graves_last_name)/\
                (jewish_graves_first_name*jewish_graves_last_name+non_jewish_graves_first_name*non_jewish_graves_last_name))
        print(f" {blank_line}", end='\r')             
        print(f"    .... First Name: {first_name:<10} • {county_name:<20} • {state_name:<15} • {county_id}  • Name Counts - Jewish/Non Jewish: {jewish_graves_first_name_frame}/{non_jewish_graves_first_name_frame}", end='\r')
        print(f" •••  Name: {last_name}, {first_name:<10} •• POJ: {poj}% •• {county_name} • {state_name} • {county_id}  • Jewish/Non Jewish Last-First: • "+\
                      f"{jewish_graves_last_name_frame}/{non_jewish_graves_last_name_frame} • {jewish_graves_first_name_frame}/{non_jewish_graves_first_name_frame}")
        name_county_counts_df=pd.DataFrame([last_name,first_name,poj,county_name,state_name,county_id,\
                                                 jewish_graves_last_name_frame,non_jewish_graves_last_name_frame,jewish_graves_first_name_frame,non_jewish_graves_first_name_frame]).T
        name_county_counts_df_header=['last_name','first_name','poj','county_name','state_name','county_id',\
                                                 'jewish_graves_last_name_frame','non_jewish_graves_last_name_frame','jewish_graves_first_name_frame','non_jewish_graves_first_name_frame']
        name_county_counts_df.columns=name_county_counts_df_header        
        name_counts_df=pd.concat([name_counts_df,name_county_counts_df])
        

        name_df.to_csv(fileOutGravesInt,mode="w", index=False,header=True)
        name_counts_df.to_csv(fileOutPOJInt,mode="w", index=False,header=True)
name_df.to_csv(fileOutGraves,mode="w", index=False,header=True)
name_counts_df.to_csv(fileOutPOJ,mode="w", index=False,header=True)
name_df.to_csv(fileOutGraves0,mode="w", index=False,header=True)
name_counts_df.to_csv(fileOutPOJ0,mode="w", index=False,header=True)
print("\n")
print(f" *** Uploaded To: {fileOutGraves0} and {fileOutPOJ0}")
print('***** Last Name Processing Complete *****' )      
   
 

**Starting ***  Extract POJ from ABMC Names   *** , Time: 2025-06-04 12:54:19.215609  **

•••• Searching: baskind,nathan • Search By: birth • Before: 1920 • Radius: 25 miles • Counties in Cluster: 6
     county_id          county_name     state_name
0  county_1910        Camden County     New Jersey
0  county_1914    Gloucester County     New Jersey
0  county_2249         Bucks County   Pennsylvania
0  county_2263      Delaware County   Pennsylvania
0  county_2286    Montgomery County   Pennsylvania
0  county_2291  Philadelphia County   Pennsylvania 

 •••  Name: baskind, nathan     •• POJ: 100% •• Bucks County •  Pennsylvania • county_2249  • Jewish/Non Jewish Last-First: • 2/0 • 56/31                                       
 •••  Name: baskind, nathan     •• POJ: 100% •• Delaware County •  Pennsylvania • county_2263  • Jewish/Non Jewish Last-First: • 1/0 • 13/1                                     
    .... Last Name:   baskind    • Philadelphia County  •  Pennsylvania   • county_2291 

In [ ]:
name_coordiantes=county_coordinates_df[['county_lat','county_lon']][county_coordinates_df['county_id']==name_county_id]
# name_location=[name_coordiantes['county_lat'],name_coordiantes['county_lon']]
name_coordiantes.iloc[0,0]

In [75]:
list(last_name_frame_df_raw)

['lastName',
 'firstName',
 'fullName',
 'birthYear',
 'deathYear',
 'birthMonth',
 'deathMonth',
 'birthDate',
 'deathDate',
 'cemeteryName',
 'cemeteryId',
 'cityName',
 'stateName',
 'maidenName',
 'Parents',
 'Children',
 'Siblings',
 'Spouses',
 'type',
 'cemetery_id',
 'cemetery_name',
 'grave_count',
 'city',
 'state',
 'stateAbbreviation',
 'county',
 'state_id',
 'county_id',
 'cemetery_lat',
 'cemetery_lon',
 'County_Cemetery_Counts']